# Cuaderno 02: Inducción, Blueprints y Cadenas `calc` — El Teorema de Nicómaco

**Objetivo:** Demostrar paso a paso que $\sum_{k=0}^n k^3 = \left(\sum_{k=0}^n k\right)^2$ aplicando dos técnicas maestras: **evitar divisiones truncadas en `Nat`** y **descomponer el problema en un *Blueprint* de 3 escalones independientes**.

## 1. Inicialización del Entorno de Verificación Lean 4

In [1]:
import pathlib, shutil, subprocess, tempfile, time

def find_lean() -> str:
    elan = shutil.which("elan") or str(pathlib.Path.home() / ".elan" / "bin" / "elan")
    if pathlib.Path(elan).is_file():
        out = subprocess.check_output([elan, "which", "lean"], text=True).strip()
        if out:
            return out
    lean = shutil.which("lean")
    if lean:
        return lean
    raise RuntimeError("No se encontró `lean` (instala elan desde https://lean-lang.org).")

LEAN_BIN = find_lean()

def run_lean(code: str, title: str = "", expect_error: bool = False) -> str:
    """Compila código Lean 4 real y muestra los estados `trace_state`, `#eval`, `#print` o errores."""
    t0 = time.perf_counter()
    with tempfile.NamedTemporaryFile("w", suffix=".lean", encoding="utf-8", delete=False) as f:
        f.write(code.strip() + "\n")
        tmp_path = pathlib.Path(f.name)
    try:
        proc = subprocess.run([LEAN_BIN, str(tmp_path)], capture_output=True, text=True, timeout=45)
        dt = (time.perf_counter() - t0) * 1000.0
        out = ((proc.stdout or "") + (proc.stderr or "")).replace(str(tmp_path), "snippet.lean").strip()
        status = "ERROR ESPERADO (DIAGNÓSTICO)" if (proc.returncode != 0 and expect_error) else (
                 "VERIFICADO POR EL KERNEL" if proc.returncode == 0 else "FALLO DE COMPILACIÓN")
        header = f"=== [{status}] {title} ({dt:.0f} ms) ==="
        print(header)
        print(out if out else "(Compilación limpia sin mensajes adicionales: 0 errores, 0 warnings)")
        print()
        if proc.returncode != 0 and not expect_error:
            raise RuntimeError(f"Lean falló inesperadamente en {title}")
        if proc.returncode == 0 and expect_error:
            raise RuntimeError(f"Se esperaba un fallo pedagógico en {title}, pero compiló con éxito")
        return out
    finally:
        tmp_path.unlink(missing_ok=True)

print("Compilador Lean activo:", LEAN_BIN)

Compilador Lean activo: /Users/legalintermedia/.elan/toolchains/leanprover--lean4---v4.36.0/bin/lean


## 2. Fase 1 del Protocolo: Especificación Ejecutable y Comprobación Empírica (`#eval`)
Antes de invertir esfuerzo en demostrar una fórmula, evaluamos ambos miembros para $n = 0, \dots, 8$.

In [2]:
spec_code = """
def sumN : Nat → Nat
  | 0     => 0
  | n + 1 => sumN n + (n + 1)

def sumCubes : Nat → Nat
  | 0     => 0
  | n + 1 => sumCubes n + (n + 1) * (n + 1) * (n + 1)

#eval List.map (fun n => (n, sumCubes n, sumN n * sumN n)) [0, 1, 2, 3, 4, 5, 6, 7, 8]
"""
_ = run_lean(spec_code, "Validación empírica de Nicómaco para n = 0..8")

=== [VERIFICADO POR EL KERNEL] Validación empírica de Nicómaco para n = 0..8 (286 ms) ===
[(0, 0, 0),
 (1, 1, 1),
 (2, 9, 9),
 (3, 36, 36),
 (4, 100, 100),
 (5, 225, 225),
 (6, 441, 441),
 (7, 784, 784),
 (8, 1296, 1296)]



## 3. Fases 2 a 6: Blueprint Completo (Gauss $\to$ Lema Algebraico $\to$ Inducción $\to$ Cancelación con `suffices`)

In [3]:
blueprint_code = """
def sumN : Nat → Nat
  | 0     => 0
  | n + 1 => sumN n + (n + 1)

def sumCubes : Nat → Nat
  | 0     => 0
  | n + 1 => sumCubes n + (n + 1) * (n + 1) * (n + 1)

-- Escalón 1: Fórmula de Gauss multiplicada por 2 (cero divisiones en Nat)
theorem two_mul_sumN (n : Nat) : 2 * sumN n = n * (n + 1) := by
  induction n with
  | zero => rfl
  | succ k ih =>
    calc
      2 * sumN (k + 1)
        = 2 * sumN k + 2 * (k + 1) := Nat.mul_add 2 (sumN k) (k + 1)
      _ = k * (k + 1) + 2 * (k + 1):= by rw [ih]
      _ = (k + 2) * (k + 1)        := (Nat.add_mul k 2 (k + 1)).symm
      _ = (k + 1) * (k + 2)        := Nat.mul_comm (k + 2) (k + 1)

-- Escalón 2: Identidad algebraica de grado 4 factorizando `s = (k + 1)²`
theorem nicomachus_step (k : Nat) :
    (k * (k + 1)) * (k * (k + 1)) + 4 * ((k + 1) * (k + 1) * (k + 1)) =
      ((k + 1) * (k + 2)) * ((k + 1) * (k + 2)) := by
  have hQuad : k * k + 4 * (k + 1) = (k + 2) * (k + 2) := by
    calc
      k * k + 4 * (k + 1)
        = k * k + 2 * k + (2 * k + 4) := by omega
      _ = k * (k + 2) + 2 * (k + 2)   := by rw [Nat.mul_add k k 2, Nat.mul_add 2 k 2]; omega
      _ = (k + 2) * (k + 2)           := (Nat.add_mul k 2 (k + 2)).symm
  have sqMul (a b : Nat) : (a * b) * (a * b) = (a * a) * (b * b) := by
    rw [Nat.mul_assoc a b (a * b), ← Nat.mul_assoc b a b, Nat.mul_comm b a,
        Nat.mul_assoc a b b, ← Nat.mul_assoc a a (b * b)]
  let s := (k + 1) * (k + 1)
  have hCube : 4 * (s * (k + 1)) = (4 * (k + 1)) * s := by
    rw [Nat.mul_comm s (k + 1), ← Nat.mul_assoc 4 (k + 1) s]
  calc
    (k * (k + 1)) * (k * (k + 1)) + 4 * ((k + 1) * (k + 1) * (k + 1))
      = (k * k) * s + (4 * (k + 1)) * s           := by rw [sqMul k (k + 1), hCube]
    _ = (k * k + 4 * (k + 1)) * s                 := (Nat.add_mul (k * k) (4 * (k + 1)) s).symm
    _ = ((k + 2) * (k + 2)) * s                   := by rw [hQuad]
    _ = s * ((k + 2) * (k + 2))                   := Nat.mul_comm _ s
    _ = ((k + 1) * (k + 2)) * ((k + 1) * (k + 2)) := (sqMul (k + 1) (k + 2)).symm

-- Escalón 3: Inducción limpia de 3 pasos gracias al Blueprint
theorem four_mul_sumCubes (n : Nat) :
    4 * sumCubes n = (n * (n + 1)) * (n * (n + 1)) := by
  induction n with
  | zero => rfl
  | succ k ih =>
    calc
      4 * sumCubes (k + 1)
        = 4 * sumCubes k + 4 * ((k + 1) * (k + 1) * (k + 1)) := Nat.mul_add 4 _ _
      _ = (k * (k + 1)) * (k * (k + 1)) + 4 * ((k + 1) * (k + 1) * (k + 1)) := by rw [ih]
      _ = ((k + 1) * (k + 2)) * ((k + 1) * (k + 2)) := nicomachus_step k

-- Teorema Final: cancelación de factor 4 mediante `suffices ... by omega`
theorem nicomachus_theorem (n : Nat) : sumCubes n = sumN n * sumN n := by
  suffices h4 : 4 * sumCubes n = 4 * (sumN n * sumN n) by
    trace_state -- Meta lineal trivial: `h4 : 4 * A = 4 * B ⊢ A = B`
    omega
  have hFour : 4 * (sumN n * sumN n) = (2 * sumN n) * (2 * sumN n) := by
    change (2 * 2) * (sumN n * sumN n) = (2 * sumN n) * (2 * sumN n)
    rw [Nat.mul_assoc 2 2 (sumN n * sumN n), ← Nat.mul_assoc 2 (sumN n) (sumN n),
        Nat.mul_comm (2 * sumN n) (sumN n), ← Nat.mul_assoc 2 (sumN n) (2 * sumN n)]
  rw [four_mul_sumCubes n, hFour, two_mul_sumN n]

#print axioms nicomachus_theorem
"""
_ = run_lean(blueprint_code, "Blueprint Completo del Teorema de Nicómaco")

=== [VERIFICADO POR EL KERNEL] Blueprint Completo del Teorema de Nicómaco (304 ms) ===
n : Nat
h4 : 4 * sumCubes n = 4 * (sumN n * sumN n)
⊢ sumCubes n = sumN n * sumN n
'nicomachus_theorem' depends on axioms: [propext, Quot.sound]



### Q&A
* **¿Por qué multiplicamos las identidades intermedias por `2` y por `4` en vez de dividir entre `2`?** Porque en `Nat` la división entera trunca residuos, mientras que multiplicar mantiene todas las ecuaciones en aritmética polinómica exacta y permite cancelar `4 * A = 4 * B → A = B` en una línea con `omega`.
* **¿Qué ventaja aporta extraer `nicomachus_step` fuera de la inducción?** Separa la complejidad algebraica (factorización de $(k+1)^2$) de la estructura inductiva, reduciendo `four_mul_sumCubes` a 3 líneas legibles en `calc`.

### Data Analysis Key Findings
* La evaluación empírica verificó la igualdad exacta en los **9 primeros casos** ($n = 0 \dots 8$, llegando hasta $\sum_{k=0}^8 k^3 = 1296 = 36^2$) antes de iniciar la prueba deductiva.
* Los **4 teoremas encadenados** del Blueprint compilaron con `0` warnings y `#print axioms` confirmó **0 axiomas externos**.

### Insights or Next Steps
* Aplica siempre el patrón `suffices hScaled : c * LHS = c * RHS by omega` cuando trabajes con sumas combinatorias o geométricas racionales sobre `Nat` o `Int`.
* Avanza al **Cuaderno 03** para ver qué ocurre cuando una inducción directa queda bloqueada por un acumulador y cómo repararla.